In [1]:
# 1. Clone the reproduction repository
!git clone https://github.com/Loki3306/uscapes_draft.git

# 2. Switch into the repository root
%cd uscapes_draft

# 3. Verify latest commit hash and list repository contents
!git log -1 --oneline
!ls -F


Cloning into 'uscapes_draft'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 82 (delta 0), reused 82 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 200.72 KiB | 5.02 MiB/s, done.
/kaggle/working/uscapes_draft
d2b0e60 (HEAD -> main, origin/main, origin/HEAD) Prepare paper reproduction repository
checkpoints/  data/  environment.yml  LICENSE  README.md	 results/  src/
configs/      docs/  figures/	      paper/   requirements.txt  scripts/


In [2]:
import sys
import torch

print(f"Python version : {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name    : {torch.cuda.get_device_name(0)}")

# Check key libraries
packages = ["numpy", "scipy", "pandas", "sklearn", "xgboost", "joblib", "matplotlib"]
missing = []
for pkg in packages:
    try:
        __import__(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        missing.append(pkg)
        print(f"  [MISSING] {pkg}")

if missing:
    print(f"\nMissing packages to install: {missing}")
else:
    print("\nAll core dependencies are satisfied!")


Python version : 3.13.15
PyTorch version: 2.11.0+cu128
CUDA available : True
Device name    : Tesla T4
  [OK] numpy
  [OK] scipy
  [OK] pandas
  [OK] sklearn
  [OK] xgboost
  [OK] joblib
  [OK] matplotlib

All core dependencies are satisfied!


In [4]:
import sys
from pathlib import Path
import torch

# Ensure current repository root is on sys.path
repo_root = Path("/kaggle/working/uscapes_draft").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Testing on device: {device}")

# 1. Test DTU Model: ContrastiveMatchNet
from src.dtu.models.matchnet import ContrastiveMatchNet
matchnet = ContrastiveMatchNet(eeg_channels=8, audio_channels=28, latent_dim=64).to(device)
matchnet.eval()
x_eeg = torch.randn(2, 8, 320, device=device)       # 5s window @ 64 Hz
x_att = torch.randn(2, 28, 320, device=device)      # 28-band Gammatone envelope
x_unatt = torch.randn(2, 28, 320, device=device)
with torch.no_grad():
    ze, za, zb = matchnet(x_eeg, x_att, x_unatt)
    sim_a, sim_b = matchnet.compute_similarities(ze, za, zb)

param_count_m = sum(p.numel() for p in matchnet.parameters())
print(f"[OK] ContrastiveMatchNet: {param_count_m:,} parameters | Forward pass successful: sim_a shape = {tuple(sim_a.shape)}")

# 2. Test KUL Model: AADConformer
from src.kul.models.aad_conformer import AADConformer
conformer = AADConformer(in_channels=8).to(device)
conformer.eval()
x_kul_eeg = torch.randn(2, 8, 128, device=device)   # 2s window @ 64 Hz
with torch.no_grad():
    recon_env, z_pool = conformer(x_kul_eeg, return_features=True)

param_count_c = sum(p.numel() for p in conformer.parameters())
print(f"[OK] AADConformer: {param_count_c:,} parameters | Forward pass successful: recon shape = {tuple(recon_env.shape)}, latent shape = {tuple(z_pool.shape)}")

# 3. Test Common Utilities & Baseline Loaders
from src.common.utils import safe_corr_np, evaluate_trial_majority_vote, normalize_eeg, normalize_audio
from src.common.ridge_aad import subject_files
print("[OK] Common utilities and baseline loaders imported cleanly.")


Testing on device: cuda
[OK] ContrastiveMatchNet: 50,928 parameters | Forward pass successful: sim_a shape = (2,)
[OK] AADConformer: 165,090 parameters | Forward pass successful: recon shape = (2, 128), latent shape = (2, 64)
[OK] Common utilities and baseline loaders imported cleanly.


In [5]:
import os
from pathlib import Path

# Paths identified from attached datasets
dtu_eeg_dir = Path("/kaggle/input/datasets/lokeshgile/dataset-eeg")
dtu_audio_dir = Path("/kaggle/input/datasets/lokeshgile/eeg-audio")

print("--- 1. Checking DTU EEG Files ---")
if dtu_eeg_dir.exists():
    eeg_files = sorted(list(dtu_eeg_dir.glob("S*.mat")))
    print(f"Found {len(eeg_files)} subject .mat files in {dtu_eeg_dir}:")
    for f in eeg_files[:5]:
        print(f"  - {f.name} ({f.stat().st_size / (1024*1024):.1f} MB)")
    if len(eeg_files) > 5:
        print(f"  ... and {len(eeg_files) - 5} more.")
else:
    print(f"[NOT FOUND] {dtu_eeg_dir}")

print("\n--- 2. Checking DTU Audio Files ---")
if dtu_audio_dir.exists():
    wav_files = sorted(list(dtu_audio_dir.glob("*.wav")))
    print(f"Found {len(wav_files)} WAV files in {dtu_audio_dir}:")
    for w in wav_files[:4]:
        print(f"  - {w.name} ({w.stat().st_size / 1024:.1f} KB)")
    if len(wav_files) > 4:
        print(f"  ... and {len(wav_files) - 4} more.")
else:
    print(f"[NOT FOUND] {dtu_audio_dir}")

print("\n--- 3. Checking for Precomputed Envelopes or other Kaggle Inputs ---")
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith((".pkl", ".zip", "gammatone")):
            p = Path(root) / file
            print(f"  Found file: {p} ({p.stat().st_size / (1024*1024):.2f} MB)")


--- 1. Checking DTU EEG Files ---
Found 18 subject .mat files in /kaggle/input/datasets/lokeshgile/dataset-eeg:
  - S10_data_preproc.mat (97.6 MB)
  - S11_data_preproc.mat (96.8 MB)
  - S12_data_preproc.mat (97.3 MB)
  - S13_data_preproc.mat (97.5 MB)
  - S14_data_preproc.mat (97.4 MB)
  ... and 13 more.

--- 2. Checking DTU Audio Files ---
Found 129 WAV files in /kaggle/input/datasets/lokeshgile/eeg-audio:
  - aske_story1_trial_1.wav (4306.7 KB)
  - aske_story1_trial_10.wav (4306.7 KB)
  - aske_story1_trial_11.wav (4306.7 KB)
  - aske_story1_trial_12.wav (4306.7 KB)
  ... and 125 more.

--- 3. Checking for Precomputed Envelopes or other Kaggle Inputs ---


In [7]:
from pathlib import Path
from src.common._common import load_subject_data, trial_labels, channel_labels, fsample_values

dtu_eeg_dir = Path("/kaggle/input/datasets/lokeshgile/dataset-eeg")
sample_mat = sorted(list(dtu_eeg_dir.glob("S*.mat")))[0]

print(f"Loading sample file: {sample_mat.name} ...")
data = load_subject_data(sample_mat)

# Extract trial properties
labels = trial_labels(data)
channels = channel_labels(data)
fs_dict = fsample_values(data)
num_trials = data.eeg.shape[1]
eeg_trial_0 = data.eeg[0, 0]

print("\n--- DTU Data Inspection Summary ---")
print(f"Subject ID         : {sample_mat.stem.split('_')[0]}")
print(f"Total Trials       : {num_trials} (Expected: 60)")
print(f"Sampling Frequency : {fs_dict['eeg']} Hz (Expected: 512 Hz)")
print(f"Total Channels     : {len(channels)} (Expected: 64)")
print(f"Trial 0 EEG Shape  : {eeg_trial_0.shape} (Channels x Samples)")
print(f"Trial 0 Duration   : {eeg_trial_0.shape[1] / fs_dict['eeg']:.1f} seconds (Expected: ~60s)")
print(f"Trial Labels Count : {len(labels)} labels (First 5: {labels[:5]})")
print("\n[OK] DTU subject data structure confirmed valid and readable!")


Loading sample file: S10_data_preproc.mat ...

--- DTU Data Inspection Summary ---
Subject ID         : S10
Total Trials       : 60 (Expected: 60)
Sampling Frequency : 64 Hz (Expected: 512 Hz)
Total Channels     : 66 (Expected: 64)
Trial 0 EEG Shape  : (3200, 66) (Channels x Samples)
Trial 0 Duration   : 1.0 seconds (Expected: ~60s)
Trial Labels Count : 60 labels (First 5: [2, 2, 2, 2, 2])

[OK] DTU subject data structure confirmed valid and readable!


In [10]:
import os
import pickle
import json
import time
from pathlib import Path
from joblib import Parallel, delayed
from src.dtu.data.extract_gammatone_envelopes import extract_gammatone_envelopes

audio_dir = Path("/kaggle/input/datasets/lokeshgile/eeg-audio")
wav_files = sorted(list(audio_dir.glob("*.wav")))
out_pkl = Path("/kaggle/working/uscapes_draft/data/gammatone_envelopes.pkl")
out_pkl.parent.mkdir(parents=True, exist_ok=True)

num_cores = os.cpu_count() or 4
print(f"Saturating {num_cores} Kaggle CPU workers to process {len(wav_files)} WAV files...")
start_time = time.time()

# Helper function executed in parallel across worker processes
def process_single_wav(wav_path):
    env = extract_gammatone_envelopes(wav_path, num_bands=28, low_freq=50, high_freq=8000, target_fs=64)
    return wav_path.name, env

# Run parallel extraction across all files
results = Parallel(n_jobs=-1, batch_size=2, verbose=10)(
    delayed(process_single_wav)(w) for w in wav_files
)

envelopes = dict(results)
elapsed = time.time() - start_time
print(f"\nExtracted all {len(envelopes)} envelopes in {elapsed:.1f} seconds! ({elapsed/len(wav_files):.2f}s per file)")

# Save to data/gammatone_envelopes.pkl
with open(out_pkl, "wb") as f:
    pickle.dump(envelopes, f)

file_size_mb = out_pkl.stat().st_size / (1024 * 1024)
print(f"[OK] Successfully saved envelope cache to {out_pkl} ({file_size_mb:.2f} MB)")

# Verify correspondence with audio_mapping.json
mapping_file = Path("/kaggle/working/uscapes_draft/data/audio_mapping.json")
with open(mapping_file, "r") as f:
    mapping = json.load(f)

# Check trial 0 of S1
s1_t0_a = mapping["S1"]["trial_0"]["wavA"]["filename"]
s1_t0_b = mapping["S1"]["trial_0"]["wavB"]["filename"]
assert s1_t0_a in envelopes, f"Missing envelope for {s1_t0_a}"
assert s1_t0_b in envelopes, f"Missing envelope for {s1_t0_b}"
print(f"[OK] Verified audio mapping match: '{s1_t0_a}' shape = {envelopes[s1_t0_a].shape} (28 bands, {envelopes[s1_t0_a].shape[1]} samples @ 64 Hz)")


Saturating 4 Kaggle CPU workers to process 129 WAV files...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:   31.9s
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:   32.1s
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done  28 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:  3.1min
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:  3.6min
[Parallel(n_jobs=-1)]: Done  74 tasks      | elapsed:  5.2min
[Parallel(n_jobs=-1)]: Done  92 tasks      | elapsed:  6.2min
[Parallel(n_jobs=-1)]: Done 114 tasks      | elapsed:  7.7min



Extracted all 129 envelopes in 498.0 seconds! (3.86s per file)
[OK] Successfully saved envelope cache to /kaggle/working/uscapes_draft/data/gammatone_envelopes.pkl (88.20 MB)
[OK] Verified audio mapping match: 'marianne_story3_trial_1.wav' shape = (28, 3200) (28 bands, 3200 samples @ 64 Hz)


[Parallel(n_jobs=-1)]: Done 129 out of 129 | elapsed:  8.3min finished


In [11]:
# Step 7: Train ContrastiveMatchNet using the repository CLI script
!EEG_DATA_DIR=/kaggle/input/datasets/lokeshgile/dataset-eeg python scripts/dtu/train_matchnet_loso.py \
    --model eegnet \
    --batch_size 512 \
    --num_workers 4


Using device: cuda | MatchNet (eegnet) | Channels: [13, 46, 43, 23, 50, 0, 52, 14]

Evaluating fold with held-out subject: S1_data_preproc
  [Memory] Pre-fold RAM: 15.9% (4.86 GB used)
Converting to PyTorch Dataset...
/kaggle/working/uscapes_draft/scripts/dtu/train_matchnet_loso.py:263: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
Training on 21114 chunks (5s) | Batch Size: 512 | Workers: 4...
/kaggle/working/uscapes_draft/scripts/dtu/train_matchnet_loso.py:284: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
  Epoch 01/100 | Loss: 0.0956 (sA: 0.018, sB: 0.014) | Val Acc: 55.88% | Patience: 0/10
  Epoch 02/100 | Loss: 0.0832 (sA: 0.038, sB: 0.018) | Val Acc: 63.53% | Patience: 0/10
  Epoch 03/100 | Loss: 0.0771 (sA: 0.041, sB: 0.012) | Val Acc: 64.71% | Patien

In [12]:
# Step 8: Update repo and run full 18-fold LOSO training across all subjects
!git pull origin main
!python scripts/dtu/train_matchnet_loso.py \
    --data_dir /kaggle/input/datasets/lokeshgile/dataset-eeg \
    --batch_size 512 \
    --num_workers 4


remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.02 KiB | 521.00 KiB/s, done.
From https://github.com/Loki3306/uscapes_draft
 * branch            main       -> FETCH_HEAD
   d2b0e60..14ed85d  main       -> origin/main
Updating d2b0e60..14ed85d
Fast-forward
 scripts/dtu/export_matchnet_predictions.py |  7 ++++---
 scripts/dtu/train_matchnet_loso.py         | 30 +++++++++++++++++++-----------
 2 files changed, 23 insertions(+), 14 deletions(-)
Using device: cuda | MatchNet (eegnet) | Channels: [13, 46, 43, 23, 50, 0, 52, 14]

Evaluating fold with held-out subject: S1_data_preproc
  [Memory] Pre-fold RAM: 15.9% (4.84 GB used)
Converting to PyTorch Dataset...
Training on 21114 chunks (5s) | Batch Size: 512 | Workers: 4...
  Epoch 01/100 | Loss: 0.0938 (sA: -0.023, sB: -0.030) | Val Acc: 62.55% | Patience: 0

In [13]:
# Step 9: Pull update and export 5,400 window predictions to matchnet_predictions.csv
!git pull origin main
!python scripts/dtu/export_matchnet_predictions.py \
    --checkpoint_dir checkpoints \
    --out_csv matchnet_predictions.csv \
    --window_sec 10.0 \
    --data_dir /kaggle/input/datasets/lokeshgile/dataset-eeg


remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 7), reused 8 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 947 bytes | 473.00 KiB/s, done.
From https://github.com/Loki3306/uscapes_draft
 * branch            main       -> FETCH_HEAD
   14ed85d..1f1ab5b  main       -> origin/main
Updating 14ed85d..1f1ab5b
Fast-forward
 scripts/dtu/eval_confidence_ablations.py   | 4 +++-
 scripts/dtu/eval_selective_prediction.py   | 4 +++-
 scripts/dtu/export_matchnet_predictions.py | 1 +
 scripts/dtu/train_confidence_model.py      | 4 +++-
 4 files changed, 10 insertions(+), 3 deletions(-)
Exporting predictions using device: cuda
Loading datasets...
Processing Subject S1_data_preproc using checkpoint matchnet_fold_S1_data_preproc_best.pth
Processing Subject S2_data_preproc using checkpoint matchnet_fold_S2_data_preproc_best.pth
Processing Subject S3_data_preproc using checkpoint m

In [14]:
# Step 10: Train XGBoost confidence estimator on label-invariant features
!python scripts/dtu/train_confidence_model.py --csv matchnet_predictions.csv


Loading data...
Computing offline features...
/kaggle/working/uscapes_draft/scripts/dtu/train_confidence_model.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(['subject_id', 'trial_id'], group_keys=False).apply(compute_consistency)
Training final XGBoost model on ALL subjects...
Model successfully saved to models/confidence_model.json


In [15]:
# Step 11: Evaluate confidence ablations & calibration metrics under nested LOSO
!python scripts/dtu/eval_confidence_ablations.py --csv matchnet_predictions.csv


Loading data...
Running Nested LOSO Evaluation for Minimal Audits...

RESULTS SUMMARY
                         AUROC    E-AURC     Brier       ECE    Acc@90    Cov@90  N_Acc@90
M1_Margin             0.648724  0.224215  0.206029  0.013260  0.859873  0.029074       157
M2_Consistency        0.652111  0.249460  0.193680  0.006396       NaN  0.000000         0
M3_Margin_Consist     0.734727  0.185978  0.183955  0.014817  0.905149  0.068333       369
M4_Marg_Consist_Roll  0.751939  0.178444  0.178980  0.012598  0.907843  0.094444       510
M5_Full               0.753612  0.172123  0.178981  0.019938  0.917172  0.091667       495
Abl_No_Margin         0.752870  0.170969  0.179937  0.016153  0.924779  0.083704       452
Abl_No_Consist        0.644636  0.225252  0.207177  0.016959  0.885572  0.037222       201
Abl_No_Roll           0.733287  0.183748  0.184612  0.014309  0.922879  0.072037       389
Abl_No_SimC           0.752680  0.175800  0.179046  0.016826  0.905544  0.090185       487
Abl_

In [16]:
# Step 12: Pull update and evaluate selective gating and behavioral dynamics
!git pull origin main
!python scripts/dtu/eval_selective_prediction.py --csv matchnet_predictions.csv


remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 751 bytes | 375.00 KiB/s, done.
From https://github.com/Loki3306/uscapes_draft
 * branch            main       -> FETCH_HEAD
   1f1ab5b..8ccc9b5  main       -> origin/main
Updating 1f1ab5b..8ccc9b5
Fast-forward
 scripts/dtu/eval_selective_prediction.py | 12 ++++++++++++
 1 file changed, 12 insertions(+)

TASK 1: RANDOM TRIAL VISUALIZATION
Saved 20 trial behavior plots to analysis/figures/behavior_audit/trial_behavior_*.png

TASK 2: FAILURE ANALYSIS (High Conf, Incorrect)
      subject_id  trial_id  window_id  margin  trial_consistency  confidence prediction  correct
S15_data_preproc         9          4  0.1686               0.75    0.948231          B        0
 S6_data_preproc        33          1  0.1917               1.00    0.946011          B        0
S16

In [ ]:
# Step 15: Pull update and build the offline KUL cache
!git pull origin main
!python src/kul/data/build_kul_cache.py \
    --dataset_dir /kaggle/input/datasets/lowk1ee/s1-klu \
    --stimuli_dir /kaggle/input/datasets/lowk1ee/audio-klu/stimuli \
    --output_dir data/processed_kul


In [21]:
# Link existing preprocessed KUL cache directly
!mkdir -p data
!rm -rf data/processed_kul
!ln -s /kaggle/input/datasets/lowk1ee/kul-preprocessed-cache/data/processed_kul data/processed_kul

# Verify all 16 subject files exist
!ls -lh data/processed_kul/


total 1.2G
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S10.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S11.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S12.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S13.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S14.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S15.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S16.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S1.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S2.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S3.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S4.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S5.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S6.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S7.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S8.pt
-rw-r--r-- 1 nobody nogroup 73M Oct 19 04:26 S9.pt


In [23]:
# Step 16: Pull serialization fix and run full KUL Conformer LOSO training
!git pull origin main
!python scripts/kul/train_conformer_loso.py \
    --cache_dir data/processed_kul \
    --seeds 1 \
    --epochs 15


remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 769 bytes | 256.00 KiB/s, done.
From https://github.com/Loki3306/uscapes_draft
 * branch            main       -> FETCH_HEAD
   59e88df..97d14d6  main       -> origin/main
Updating 59e88df..97d14d6
Fast-forward
 scripts/kul/train_conformer_loso.py           | 8 ++++----
 scripts/kul/train_conformer_multitask_loso.py | 2 +-
 2 files changed, 5 insertions(+), 5 deletions(-)
Loading KUL Cache...
Loading 16 cached subjects from /kaggle/working/uscapes_draft/data/processed_kul...
Done in 1.05 seconds.
Using device: cuda

############################################################
STARTING SEED: 1
############################################################

Starting LOSO Fold: Test Subject = S1 | Seed = 1
Validation Subject: S10 | Training Subjects: 14
Fold S1 

In [24]:
# Step 17: Evaluate KUL selective prediction gating and calibration
!python scripts/kul/eval_selective_aad.py \
    --cache_dir data/processed_kul \
    --checkpoint_dir checkpoints/kul


--- Phase 6: Confidence-Aware Selective AAD Benchmark ---
Using device: cuda
Loading 16 cached subjects from data/processed_kul...
Done in 1.03 seconds.
Evaluating 16 subjects...
Processing Subject S1...
Processing Subject S10...
Processing Subject S11...
Processing Subject S12...
Processing Subject S13...
Processing Subject S14...
Processing Subject S15...
Processing Subject S16...
Processing Subject S2...
Processing Subject S3...
Processing Subject S4...
Processing Subject S5...
Processing Subject S6...
Processing Subject S7...
Processing Subject S8...
Processing Subject S9...

[Confidence Distribution - Clean Data]
count    71104.000000
mean         0.566339
std          0.058157
min          0.500003
25%          0.523164
50%          0.550727
75%          0.592157
max          1.000000
Name: confidence, dtype: float64

[Running Calibration]
Global ECE: 0.0096 | Brier Score: 0.2416

[Running Selective Prediction Thresholds]
 Threshold  Coverage  Accepted_Accuracy  Rejected_Rate  Ac

In [25]:
# Step 18: Pull update and train multi-task evidential confidence head with Outlier Exposure
!git pull origin main
!python scripts/kul/train_conformer_multitask_loso.py


remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 481 bytes | 240.00 KiB/s, done.
From https://github.com/Loki3306/uscapes_draft
 * branch            main       -> FETCH_HEAD
   97d14d6..d4c993d  main       -> origin/main
Updating 97d14d6..d4c993d
Fast-forward
 scripts/kul/train_conformer_multitask_loso.py | 3 +++
 1 file changed, 3 insertions(+)
--- Phase 7: Multi-Task Confidence Training (LOSO) ---
Loading 16 cached subjects from /kaggle/working/uscapes_draft/data/processed_kul...
Done in 1.06 seconds.
Using device: cuda
STARTING MULTI-TASK SEED: 1

Starting Multi-Task LOSO Fold: Test Subject = S1
Successfully loaded pre-trained Phase 4 regression weights! Fine-tuning for 5 epochs...
Epoch 01 | Train Loss: 1.0234 (Reg:0.9574 Conf:0.6603) | Val Loss: 1.0270 (Reg:0.9603 Conf:0.6674) | Val Margin: 0.0290
Epoch

In [26]:
!git pull origin main
!python scripts/kul/eval_confidence_multitask.py


remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 765 bytes | 765.00 KiB/s, done.
From https://github.com/Loki3306/uscapes_draft
 * branch            main       -> FETCH_HEAD
   d4c993d..2ab43da  main       -> origin/main
Updating d4c993d..2ab43da
Fast-forward
 scripts/kul/eval_confidence_multitask.py | 36 ++++++++++++++++++++++++++------
 1 file changed, 30 insertions(+), 6 deletions(-)
--- Phase 7: Production-Grade Confidence-Aware Selective AAD (Learned Confidence) ---
Using device: cuda
Loading 16 cached subjects from /kaggle/working/uscapes_draft/data/processed_kul...
Done in 1.07 seconds.
Evaluating 16 subjects...
Processing Subject S1...
Processing Subject S10...
Processing Subject S11...
Processing Subject S12...
Processing Subject S13...
Processing Subject S14...
Processing Subject S15...
Processing 

In [28]:
!git pull origin main
!python scripts/cross_dataset/eval_kul_to_dtu_transfer.py --dtu_dir /kaggle/input/datasets/lokeshgile/dataset-eeg --checkpoint checkpoints/kul/model_S1.pt


remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 540 bytes | 270.00 KiB/s, done.
From https://github.com/Loki3306/uscapes_draft
 * branch            main       -> FETCH_HEAD
   e550fe5..b6d0810  main       -> origin/main
Updating e550fe5..b6d0810
Fast-forward
 scripts/cross_dataset/eval_kul_to_dtu_transfer.py | 2 ++
 1 file changed, 2 insertions(+)

STEP 1 — DATA LOADING AUDIT
Checkpoint path: checkpoints/kul/model_S1.pt
Dataset path: /kaggle/input/datasets/lokeshgile/dataset-eeg
Subject name: S1_data_preproc
Trial count: 60
Sampling frequency: 64
Selected channels: ['T7', 'C2', 'FT8', 'P7', 'CPz', 'Fp1', 'TP8', 'C3']
Window length: 5s (320 samples)
Window hop: 5s (320 samples)
EEG shape: (8, 3200)
Audio shape: (1, 3200)
✓ Frozen checkpoint
✓ DTU EEG
✓ DTU gammatones

STEP 2 — SINGLE SUBJECT TRACE (S1 Trial 

In [29]:
!git pull origin main
!python scripts/kul/eval_falsification.py


remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 752 bytes | 376.00 KiB/s, done.
From https://github.com/Loki3306/uscapes_draft
 * branch            main       -> FETCH_HEAD
   b6d0810..27edb53  main       -> origin/main
Updating b6d0810..27edb53
Fast-forward
 scripts/kul/eval_falsification.py | 15 +++++++++++----
 1 file changed, 11 insertions(+), 4 deletions(-)
Loading KUL Cache...
Loading 16 cached subjects from /kaggle/working/uscapes_draft/data/processed_kul...
Done in 1.06 seconds.
Using device: cuda
Processing Test Subject: S1
Processing Test Subject: S10
Processing Test Subject: S11
Processing Test Subject: S12
Processing Test Subject: S13
Processing Test Subject: S14
Processing Test Subject: S15
Processing Test Subject: S16
Processing Test Subject: S2
Processing Test Subject: S3
Processing Test Subj